In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.pmod.pmod_io import Pmod_IO
import time
base = BaseOverlay("base.bit")
base.select_pmoda()
IN1 = Pmod_IO(base.PMODA, 3, 'out')
IN2 = Pmod_IO(base.PMODA, 2, 'out')
IN3 = Pmod_IO(base.PMODA, 1, 'out')
IN4 = Pmod_IO(base.PMODA, 0, 'out')
def stop():
    for p in (IN1, IN2, IN3, IN4): p.write(0)
def motor_a(d): IN1.write(1 if d > 0 else 0); IN2.write(1 if d < 0 else 0)
def motor_b(d): IN3.write(1 if d > 0 else 0); IN4.write(1 if d < 0 else 0)
stop()
print("初始化完成")

In [ ]:
SIGN_A = 1
SIGN_B = 1

def drive(a, b):
    motor_a(a * SIGN_A)
    motor_b(b * SIGN_B)

try:
    drive(1, 1);   time.sleep(2); drive(0, 0); time.sleep(1)   # 期望:两轮同向
    drive(-1, -1); time.sleep(2); drive(0, 0); time.sleep(1)   # 期望:两轮同向,与上一次相反
    drive(-1, 1);  time.sleep(2); drive(0, 0)                  # 期望:一前一后,原地转向
finally:
    stop()

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.pmod.pmod_iic import Pmod_IIC
from pynq.lib.pmod.pmod_io import Pmod_IO
import time, numpy as np

base = BaseOverlay("base.bit")
base.select_pmoda()

iic = Pmod_IIC(base.PMODB, 6, 7, 0x68)
iic.send([0x6B, 0x00]); time.sleep(0.1)

IN1 = Pmod_IO(base.PMODA, 3, 'out')
IN2 = Pmod_IO(base.PMODA, 2, 'out')
IN3 = Pmod_IO(base.PMODA, 1, 'out')
IN4 = Pmod_IO(base.PMODA, 0, 'out')
def stop():
    for p in (IN1, IN2, IN3, IN4): p.write(0)
def motor_a(d): IN1.write(1 if d > 0 else 0); IN2.write(1 if d < 0 else 0)
def motor_b(d): IN3.write(1 if d > 0 else 0); IN4.write(1 if d < 0 else 0)
SIGN_A, SIGN_B = 1, 1
def drive(a, b): motor_a(a * SIGN_A); motor_b(b * SIGN_B)
stop()

def s16(hi, lo):
    v = (hi << 8) | lo
    return v - 65536 if v & 0x8000 else v

def read_imu():
    iic.send([0x3B])
    d = iic.receive(14)
    ax, ay, az = [s16(d[i], d[i+1]) / 16384.0 for i in (0, 2, 4)]
    gx, gy, gz = [s16(d[i], d[i+1]) / 131.0 for i in (8, 10, 12)]
    return ax, ay, az, gx, gy, gz

print("静止校准中,请不要碰车...")
time.sleep(1)
cal = np.array([read_imu() for _ in range(300)])
bias = cal[:, 3:6].mean(axis=0)
print("陀螺仪零偏 (X,Y,Z) °/s:", bias.round(3))
print("加速度均值 (g):", cal[:, 0:3].mean(axis=0).round(3))

In [ ]:
import csv

def run_and_record(label, a, b, dur):
    rows = []
    t0 = time.time()
    drive(a, b)
    try:
        while time.time() - t0 < dur:
            rows.append((time.time() - t0,) + read_imu())
    finally:
        stop()
    time.sleep(0.8)
    r = np.array(rows)
    t = r[:, 0]; g = r[:, 4:7] - bias; acc = r[:, 1:4]
    dt = np.diff(t, prepend=t[0])
    ang = (g * dt[:, None]).sum(axis=0)
    print(f"[{label}] 样本数={len(r)}  实际速率≈{len(r)/dur:.0f}Hz")
    print("  陀螺仪均值(X,Y,Z) °/s:", g.mean(axis=0).round(2))
    print("  陀螺仪积分角度(X,Y,Z) °:", ang.round(1))
    print("  加速度标准差(g):", acc.std(axis=0).round(3))
    with open(f"imu_{label}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["t", "ax", "ay", "az", "gx", "gy", "gz"])
        w.writerows(r.tolist())
    return r

In [ ]:
r_a = run_and_record("A_straight_air", 1, 1, 3)

In [ ]:
r_left  = run_and_record("B_spin_a", -1, 1, 1.0)
r_right = run_and_record("B_spin_b", 1, -1, 1.0)

In [ ]:
SIGN_A, SIGN_B = -1, -1     # 改成 -1 后 drive(1,1) = 前进

def recalibrate(n=300):
    global bias
    time.sleep(1)
    cal = np.array([read_imu() for _ in range(n)])
    bias = cal[:, 3:6].mean(axis=0)
    print("零偏:", bias.round(3))

def turn_gyro(deg, timeout=5.0):
    """deg>0 用 drive(-1,1),陀螺仪Z轴积分到 +deg 就停;deg<0 反向"""
    direction = 1 if deg > 0 else -1
    target = abs(deg)
    ang = 0.0
    t_prev = t_start = time.time()
    drive(-direction, direction)
    try:
        while abs(ang) < target and time.time() - t_start < timeout:
            r = read_imu()
            t = time.time()
            ang += (r[5] - bias[2]) * (t - t_prev)
            t_prev = t
    finally:
        stop()
    ang_at_stop = ang
    t_prev = time.time(); t_end = t_prev + 1.0
    while time.time() < t_end:
        r = read_imu()
        t = time.time()
        ang += (r[5] - bias[2]) * (t - t_prev)
        t_prev = t
    print(f"目标 {deg}°  停车时 {ang_at_stop:.1f}°  滑行后总计 {ang:.1f}°")
    return ang

recalibrate()

In [ ]:
drive(1, 1); time.sleep(1); stop()

In [ ]:
for d in (90, -90, 90, -90):
    recalibrate(150)
    turn_gyro(d)
    time.sleep(2)

In [ ]:
def stop():
    for p in (IN1, IN2, IN3, IN4): p.write(1)
    time.sleep(0.15)
    for p in (IN1, IN2, IN3, IN4): p.write(0)

for d in (90, -90, 90, -90):
    recalibrate(150)
    turn_gyro(d)
    time.sleep(2)

In [ ]:
import pynq; print(pynq.__version__)

In [ ]:
import socket, time, struct, numpy as np

PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 1000

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.settimeout(0.2)
rtts, lost = [], 0
for i in range(N):
    pkt = struct.pack("!I", i) + bytes(60)
    t0 = time.perf_counter()
    s.sendto(pkt, (PC_IP, 9000))
    try:
        while True:
            d, _ = s.recvfrom(2048)
            if struct.unpack("!I", d[:4])[0] == i:
                break
        rtts.append((time.perf_counter() - t0) * 1000)
    except socket.timeout:
        lost += 1
    time.sleep(0.01)

r = np.array(rtts)
print(f"发送 {N}  丢包 {lost} ({lost/N*100:.2f}%)")
print(f"往返延迟 ms:  P50={np.percentile(r,50):.2f}  P95={np.percentile(r,95):.2f}  "
      f"P99={np.percentile(r,99):.2f}  max={r.max():.2f}")
np.savetxt("udp_rtt_baseline.csv", r, header="rtt_ms", fmt="%.4f")

In [ ]:
import cv2, socket, struct, time, numpy as np
PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 300

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)     # 不解码,拿原始数据
for _ in range(10): cap.read()

ok, f = cap.read()
print("原始帧形状:", f.shape, f.dtype, " 前两字节:", bytes(f.flatten()[:2]).hex())
# 如果前两字节是 ffd8,说明是 JPEG 原始数据,可以继续

s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((PC_IP, 9001))

def recvn(n):
    buf = b""
    while len(buf) < n:
        c = s.recv(n - len(buf))
        if not c: raise ConnectionError
        buf += c
    return buf

cap_ms, net_ms, tot_ms, sizes = [], [], [], []
T0 = time.perf_counter()
for i in range(N):
    t0 = time.perf_counter()
    ok, frame = cap.read()
    t1 = time.perf_counter()
    data = frame.tobytes()
    s.sendall(struct.pack("!II", i, len(data)) + data)
    recvn(4)
    t2 = time.perf_counter()
    cap_ms.append((t1-t0)*1000); net_ms.append((t2-t1)*1000)
    tot_ms.append((t2-t0)*1000); sizes.append(len(data))
T = time.perf_counter() - T0
s.close(); cap.release()

def p(a): return f"P50={np.percentile(a,50):.1f} P95={np.percentile(a,95):.1f} P99={np.percentile(a,99):.1f} max={max(a):.1f}"
print(f"帧数 {N}  平均帧大小 {np.mean(sizes)/1024:.1f} KB  吞吐 {sum(sizes)*8/T/1e6:.2f} Mbps  帧率 {N/T:.1f} FPS")
print("采集 ms:", p(cap_ms))
print("网络+回执 ms:", p(net_ms))
print("端到端 ms:", p(tot_ms))
np.savetxt("img_stream_rawmjpg.csv", np.c_[cap_ms, net_ms, tot_ms, sizes],
           delimiter=",", header="cap_ms,net_ms,total_ms,bytes", fmt="%.4f")

In [ ]:
import os
os.system("ifconfig")

In [ ]:
import cv2, socket, struct, time, numpy as np
PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 200

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
for _ in range(10): cap.read()

s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((PC_IP, 9001))

def recvn(n):
    buf = b""
    while len(buf) < n:
        c = s.recv(n - len(buf))
        if not c: raise ConnectionError
        buf += c
    return buf

tot_ms, inf_ms_l, rows = [], [], []
T0 = time.perf_counter()
for i in range(N):
    t0 = time.perf_counter()
    ok, frame = cap.read()
    data = frame.tobytes()
    s.sendall(struct.pack("!II", i, len(data)) + data)
    fid, cnt, cx, area, inf = struct.unpack("!IIfff", recvn(20))
    t1 = time.perf_counter()
    tot_ms.append((t1 - t0) * 1000); inf_ms_l.append(inf)
    rows.append((i, cnt, cx, area, inf, (t1 - t0) * 1000))
T = time.perf_counter() - T0
s.close(); cap.release()

def p(a): return f"P50={np.percentile(a,50):.1f} P95={np.percentile(a,95):.1f} P99={np.percentile(a,99):.1f} max={max(a):.1f}"
print(f"帧率 {N/T:.1f} FPS")
print("PC 端推理 ms:", p(inf_ms_l))
print("端到端 ms:", p(tot_ms))
det = [r for r in rows if r[1] > 0]
print(f"检测到物体的帧: {len(det)}/{N}")
np.savetxt("yolo_pipeline_baseline.csv", np.array(rows), delimiter=",",
           header="frame,count,cx,area,infer_ms,total_ms", fmt="%.4f")

In [ ]:
import cv2, socket, struct, time, numpy as np, csv
PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 300

CMD = {"forward": (1, 1), "left": (-1, 1), "right": (1, -1)}

def decide(cnt, cx, area):
    if cnt == 0 or area < 0.10:
        return "forward"
    return "right" if cx < 0.5 else "left"   # 障碍在画面左→右转,在右→左转

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
for _ in range(10): cap.read()

s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((PC_IP, 9001))

def recvn(n):
    buf = b""
    while len(buf) < n:
        c = s.recv(n - len(buf))
        if not c: raise ConnectionError
        buf += c
    return buf

lat, log = [], []
try:
    for i in range(N):
        ok, frame = cap.read()
        t_cap = time.perf_counter()               # 帧已拿到
        data = frame.tobytes()
        s.sendall(struct.pack("!II", i, len(data)) + data)
        fid, cnt, cx, area, inf = struct.unpack("!IIfff", recvn(20))
        act = decide(cnt, cx, area)
        a, b = CMD[act]
        drive(a, b)
        t_act = time.perf_counter()               # 电机指令已写出
        lat.append((t_act - t_cap) * 1000)
        log.append((i, cnt, round(cx, 3), round(area, 3), round(inf, 2), act, round(lat[-1], 2)))
finally:
    stop()
    s.close(); cap.release()

def p(a): return f"P50={np.percentile(a,50):.1f} P95={np.percentile(a,95):.1f} P99={np.percentile(a,99):.1f} max={max(a):.1f}"
print("拿到画面 → 电机指令写出 ms:", p(lat))
from collections import Counter
print("动作统计:", Counter(r[5] for r in log))
with open("closed_loop_dryrun.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "count", "cx", "area", "infer_ms", "action", "latency_ms"])
    w.writerows(log)

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.pmod.pmod_io import Pmod_IO
import time

base = BaseOverlay("base.bit")
base.select_pmoda()
IN1 = Pmod_IO(base.PMODA, 3, 'out')
IN2 = Pmod_IO(base.PMODA, 2, 'out')
IN3 = Pmod_IO(base.PMODA, 1, 'out')
IN4 = Pmod_IO(base.PMODA, 0, 'out')

SIGN_A, SIGN_B = -1, -1     # drive(1,1) = 前进

def motor_a(d): IN1.write(1 if d > 0 else 0); IN2.write(1 if d < 0 else 0)
def motor_b(d): IN3.write(1 if d > 0 else 0); IN4.write(1 if d < 0 else 0)
def drive(a, b): motor_a(a * SIGN_A); motor_b(b * SIGN_B)

def stop():
    for p in (IN1, IN2, IN3, IN4): p.write(1)     # 先刹车
    time.sleep(0.15)
    for p in (IN1, IN2, IN3, IN4): p.write(0)

for p in (IN1, IN2, IN3, IN4): p.write(0)
print("电机初始化完成")

In [ ]:
import cv2, socket, struct, time, numpy as np, csv
PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 300

CMD = {"forward": (1, 1), "left": (-1, 1), "right": (1, -1)}

def decide(cnt, cx, area):
    if cnt == 0 or area < 0.10:
        return "forward"
    return "right" if cx < 0.5 else "left"   # 障碍在画面左→右转,在右→左转

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
for _ in range(10): cap.read()

s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((PC_IP, 9001))

def recvn(n):
    buf = b""
    while len(buf) < n:
        c = s.recv(n - len(buf))
        if not c: raise ConnectionError
        buf += c
    return buf

lat, log = [], []
try:
    for i in range(N):
        ok, frame = cap.read()
        t_cap = time.perf_counter()               # 帧已拿到
        data = frame.tobytes()
        s.sendall(struct.pack("!II", i, len(data)) + data)
        fid, cnt, cx, area, inf = struct.unpack("!IIfff", recvn(20))
        act = decide(cnt, cx, area)
        a, b = CMD[act]
        drive(a, b)
        t_act = time.perf_counter()               # 电机指令已写出
        lat.append((t_act - t_cap) * 1000)
        log.append((i, cnt, round(cx, 3), round(area, 3), round(inf, 2), act, round(lat[-1], 2)))
finally:
    stop()
    s.close(); cap.release()

def p(a): return f"P50={np.percentile(a,50):.1f} P95={np.percentile(a,95):.1f} P99={np.percentile(a,99):.1f} max={max(a):.1f}"
print("拿到画面 → 电机指令写出 ms:", p(lat))
from collections import Counter
print("动作统计:", Counter(r[5] for r in log))
with open("closed_loop_dryrun.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "count", "cx", "area", "infer_ms", "action", "latency_ms"])
    w.writerows(log)

In [ ]:
import cv2, socket, struct, time, numpy as np, csv
from collections import Counter
PC_IP = "192.168.2.1"      # 改成你电脑的 IP
N = 300
BAD = 0xFFFFFFFF

CMD = {"forward": (1, 1), "left": (-1, 1), "right": (1, -1)}

def decide(cnt, cx, area):
    if cnt == 0 or area < 0.10:
        return "forward"
    return "right" if cx < 0.5 else "left"

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
for _ in range(30): cap.read()          # 多丢几帧,让摄像头稳定

s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((PC_IP, 9001))

def recvn(n):
    buf = b""
    while len(buf) < n:
        c = s.recv(n - len(buf))
        if not c: raise ConnectionError
        buf += c
    return buf

lat, log = [], []
bad_local, bad_remote = 0, 0
try:
    for i in range(N):
        ok, frame = cap.read()
        t_cap = time.perf_counter()
        data = frame.tobytes() if ok else b""
        if data[:2] != b"\xff\xd8":              # 不是 JPEG 开头,丢弃
            bad_local += 1
            continue
        s.sendall(struct.pack("!II", i, len(data)) + data)
        fid, cnt, cx, area, inf = struct.unpack("!IIfff", recvn(20))
        if cnt == BAD:                           # 服务器说这帧解码失败
            bad_remote += 1
            continue
        act = decide(cnt, cx, area)
        a, b = CMD[act]
        drive(a, b)
        t_act = time.perf_counter()
        lat.append((t_act - t_cap) * 1000)
        log.append((i, cnt, round(cx, 3), round(area, 3), round(inf, 2), act, round(lat[-1], 2)))
finally:
    stop()
    s.close(); cap.release()

def p(a): return f"P50={np.percentile(a,50):.1f} P95={np.percentile(a,95):.1f} P99={np.percentile(a,99):.1f} max={max(a):.1f}"
print(f"有效帧 {len(log)}/{N}  本地丢弃 {bad_local}  服务器解码失败 {bad_remote}")
print("拿到画面 → 电机指令写出 ms:", p(lat))
print("动作统计:", Counter(r[5] for r in log))
with open("closed_loop_dryrun.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "count", "cx", "area", "infer_ms", "action", "latency_ms"])
    w.writerows(log)